In [1]:
import torch
from torch  import nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report,cohen_kappa_score
import time
from model import base_class
from models import utils,construt_tree,MFS_HiNet,TreeRep
import data_reader
from random import randint
from collections import Counter
import split_data
import random
import os
from models import create_graph
import torch.utils.checkpoint as checkpoint
from sklearn import metrics, preprocessing
import torch.nn.functional as F

/data/yao.jin/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calculate_metrics(predictions, labels, num_classes):
    # 计算混淆矩阵
    cm = confusion_matrix(labels, predictions, labels=np.arange(num_classes))
    
    # 计算OA（总体精度）
    OA = np.trace(cm) / np.sum(cm)

    # 计算AA（均匀精度）
    AA = np.mean(np.diag(cm) / np.sum(cm, axis=1))

    # 计算Kappa系数
    total_pixels = np.sum(cm)
    P_o = np.trace(cm) / total_pixels  # 观察到的精度
    P_e = np.sum(np.sum(cm, axis=0) * np.sum(cm, axis=1)) / (total_pixels ** 2)  # 期望精度
    Kappa = (P_o - P_e) / (1 - P_e)

    # 计算每个类别的精度和召回率
    precision = np.diag(cm) / np.sum(cm, axis=0)  # 精度 = TP / (TP + FP)
    recall = np.diag(cm) / np.sum(cm, axis=1)  # 召回率 = TP / (TP + FN)
    
    # 计算类别的精度和召回率时要防止除以零
    precision = np.nan_to_num(precision, nan=0.0)  # 防止除以零
    recall = np.nan_to_num(recall, nan=0.0)  # 防止除以零

    return OA, AA, Kappa, precision, recall

In [28]:
def evaluate_performance_all(

        network_output,
        train_samples_gt,
        train_samples_gt_onehot,

        m,
        n,
        class_count,

        Test_GT,

        require_AA_KPP=False,
        printFlag=True

):

    """
    说明
    --------------------------------------------------------
    原始GT:
        0 = background
        1~C = real class

    train_samples_gt:
        已经过滤背景并重新编码
        0~C-1

    因此：

    valid_mask 必须来自原始 Test_GT

    而：

    gt_label 来自 onehot argmax
    --------------------------------------------------------
    """

    OA_ALL = []
    AA_ALL = []
    KPP_ALL = []
    AVG_ALL = []

    with torch.no_grad():

        # =====================================================
        # GT label（0-based）
        # =====================================================

        gt_label = torch.argmax(

            train_samples_gt_onehot,
            dim=1

        )

        # =====================================================
        # valid mask
        #
        # 原始GT:
        # 0 = background
        # =====================================================

        if isinstance(Test_GT, torch.Tensor):

            valid_mask = (
                Test_GT.reshape(-1) != 0
            ).float().to(network_output.device)

            Test_GT_np = (
                Test_GT.cpu().numpy()
            )

        else:

            valid_mask = torch.from_numpy(

                (
                    Test_GT.reshape(-1) != 0
                ).astype(np.float32)

            ).to(network_output.device)

            Test_GT_np = Test_GT

        valid_count = valid_mask.sum()

        # =====================================================
        # OA
        # =====================================================

        correct_prediction = torch.where(

            network_output == gt_label,

            valid_mask,

            torch.zeros_like(valid_mask)

        ).sum()

        OA = (
            correct_prediction.cpu()
            / valid_count.cpu()
        ).numpy()

        # =====================================================
        # only OA
        # =====================================================

        if require_AA_KPP is False:

            return OA

        # =====================================================
        # numpy
        # =====================================================

        pred = network_output.cpu().numpy()

        gt = gt_label.cpu().numpy()

        valid_mask_np = (
            Test_GT_np.reshape(-1) != 0
        )

        # =====================================================
        # AA
        # =====================================================

        count_perclass = np.zeros(class_count)

        correct_perclass = np.zeros(class_count)

        for cls in range(class_count):

            cls_mask = (
                gt == cls
            )

            cls_count = cls_mask.sum()

            count_perclass[cls] = cls_count

            if cls_count > 0:

                cls_correct = (

                    pred[cls_mask]
                    == gt[cls_mask]

                ).sum()

                correct_perclass[cls] = cls_correct

        test_AC_list = np.divide(

            correct_perclass,

            count_perclass,

            out=np.zeros_like(correct_perclass),

            where=count_perclass != 0

        )

        test_AA = np.mean(test_AC_list)

        # =====================================================
        # Kappa
        # =====================================================

        test_pre_label_list = pred[
            valid_mask_np
        ]

        test_real_label_list = gt[
            valid_mask_np
        ]

        test_kpp = metrics.cohen_kappa_score(

            test_real_label_list.astype(np.int16),

            test_pre_label_list.astype(np.int16)

        )

        # =====================================================
        # print
        # =====================================================

        if printFlag:

            print(

                "test OA =",
                OA,

                "AA =",
                test_AA,

                "kpp =",
                test_kpp

            )

            print('acc per class:')

            print(test_AC_list)

        OA_ALL.append(OA)

        AA_ALL.append(test_AA)

        KPP_ALL.append(test_kpp)

        AVG_ALL.append(test_AC_list)

        return (

            OA,
            OA_ALL,
            AA_ALL,
            KPP_ALL,
            AVG_ALL

        )

In [4]:
np.set_printoptions(threshold=np.inf)
torch.set_printoptions(sci_mode=False)

In [5]:

def setup_seed(seed):
    random.seed(seed)  # Python的随机性
    os.environ['PYTHONHASHSEED'] = str(seed)  # 设置Python哈希种子，为了禁止hash随机化，使得实验可复现
    np.random.seed(seed)  # numpy的随机性
    torch.manual_seed(seed)  # torch的CPU随机性，为CPU设置随机种子
    torch.cuda.manual_seed(seed)  # torch的GPU随机性，为当前GPU设置随机种子
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.   torch的GPU随机性，为所有GPU设置随机种子
    torch.backends.cudnn.deterministic = True # 选择确定性算法a
    torch.backends.cudnn.benchmark = False # if benchmark=True, deterministic will be False

In [6]:
def evaluate_performance(network_output, train_samples_gt, train_samples_gt_onehot, zeros):
    with torch.no_grad():
        available_label_idx = (train_samples_gt!=0).float()        # 有效标签的坐标,用于排除背景
        available_label_count = available_label_idx.sum()          # 有效标签的个数
        correct_prediction = torch.where(network_output ==torch.argmax(train_samples_gt_onehot, 1), available_label_idx, zeros).sum()
        OA= correct_prediction.cpu() / available_label_count
        return OA

In [7]:
def load_data():
    data = data_reader.PaviaURaw().normal_cube
    data_gt = data_reader.PaviaURaw().truth
    return data, data_gt

In [8]:
patch_size = 19
pca_components = 32
split_type = ['number', 'ratio'][0]
train_num=20#划分训练集每类的个数
val_num =20
train_ratio = 0.001# 划分训练集每类的比例
val_ratio = 0.001#0 测试集比例.注意，验证集选取为从测试集整体随机选取，非按照每类,所以可能有的类不再测试集中

max_epoch = 500
batch_size = 128
learning_rate = 0.0005 # 学习率
dataset_name = 'indian_'
# dataset_name = "pavia_"
from models.utils import setup_seed
path_weight = r"/data/yao.jin/CNN/dataset/weights//"
path_result = r"/data/yao.jin/CNN/dataset/result//"
data, data_gt = load_data()
height, width, bands = data.shape
gt_reshape = np.reshape(data_gt, [-1])
class_num = np.max(data_gt)
class_num = class_num.astype(int)
seed = randint(1,9999)
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [9]:
train_index, val_index, test_index = split_data.split_data(gt_reshape, 
                class_num, train_ratio, train_ratio, train_num, val_num, split_type)
train_index=train_index.astype(int)
val_index=val_index.astype(int)
test_index=test_index.astype(int)

In [10]:
class_num = np.max(data_gt)
class_num = class_num.astype(int)
gt_reshape = np.reshape(data_gt, [-1])
height, width, bands = data.shape
train_samples_gt, test_samples_gt, val_samples_gt = create_graph.get_label(gt_reshape,
                                                train_index, val_index, test_index)

train_label_mask, test_label_mask, val_label_mask = create_graph.get_label_mask(train_samples_gt, 
                                        test_samples_gt, val_samples_gt, data_gt, class_num)

# label transfer to one-hot encode




In [11]:
#setup_seed(9085)
class_num = np.max(data_gt)
class_num = class_num.astype(int)
gt_reshape = np.reshape(data_gt, [-1])
height, width, bands = data.shape
train_samples_gt, test_samples_gt, val_samples_gt = create_graph.get_label(gt_reshape,
                                                train_index, val_index, test_index)

train_label_mask, test_label_mask, val_label_mask = create_graph.get_label_mask(train_samples_gt, 
                                        test_samples_gt, val_samples_gt, data_gt, class_num)

# label transfer to one-hot encode
train_gt = np.reshape(train_samples_gt,[height,width])
test_gt = np.reshape(test_samples_gt,[height,width])
val_gt = np.reshape(val_samples_gt,[height,width])


train_gt_onehot = create_graph.label_to_one_hot(train_gt, class_num)
test_gt_onehot = create_graph.label_to_one_hot(test_gt, class_num)
val_gt_onehot = create_graph.label_to_one_hot(val_gt, class_num)



train_samples_gt=torch.from_numpy(train_samples_gt.astype(np.int32)).to(device)
test_samples_gt=torch.from_numpy(test_samples_gt.astype(np.int32)).to(device)
val_samples_gt=torch.from_numpy(val_samples_gt.astype(np.int32)).to(device)
train_gt_onehot = torch.from_numpy(train_gt_onehot.astype(np.int32)).to(device)
test_gt_onehot = torch.from_numpy(test_gt_onehot.astype(np.int32)).to(device)
val_gt_onehot = torch.from_numpy(val_gt_onehot.astype(np.int32)).to(device)

train_label_mask = torch.from_numpy(train_label_mask.astype(np.int32)).to(device)
test_label_mask = torch.from_numpy(test_label_mask.astype(np.int32)).to(device)
val_label_mask = torch.from_numpy(val_label_mask.astype(np.int32)).to(device)
time9 = time.time()

In [12]:
test_samples_gt.shape

torch.Size([207400])

In [13]:
hi_margin = 5
euc_margin = 5
layer=3

In [14]:
kernel_size=5

In [15]:
net_input=np.array(data, np.float32)
net_input=torch.from_numpy(net_input.astype(np.float32)).to(device)
model = MFS_HiNet.BackboneHi(bands, class_num,layer,0.5,kernel_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.999), eps=1e-08, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 15, eta_min=0.0, last_epoch=-1)
criterion = nn.CrossEntropyLoss()
zeros = torch.zeros([height * width]).to(device).float()
best_loss=99999
model.train()
tic1 = time.time()

In [16]:
for i in range(500+1):
    optimizer.zero_grad()  # zero the gradient buffers
    output_hi,c= model(net_input)
    #random_integers = np.random.randint(low=0, high=799, size=320)
    #hi_loss = utils.bolicDistance_triloss(output_hi[train_index[random_integers]],train_samples_gt[train_index[random_integers]],c,hi_margin)
    #euc_loss = utils.eucDistance_triloss(output_euc[train_index[random_integers]],train_samples_gt[train_index[random_integers]],euc_margin)
    hi_loss = utils.bolicDistance_triloss(output_hi[train_index],train_samples_gt[train_index],c,hi_margin)
    #euc_loss = utils.eucDistance_triloss(output_euc[train_index],train_samples_gt[train_index],euc_margin)
    loss = hi_loss
    #loss =(loss-b).abs()+b
    loss.backward(retain_graph=True)
    #euc_loss.backward(retain_graph=True)
    optimizer.step()  # Does the update
    
    if i%10==0:
        with torch.no_grad():
            model.eval()
            optimizer.zero_grad()  # zero the gradient buffers
            output_hi,c= model(net_input)
            val_hi_loss = utils.bolicDistance_triloss(output_hi[val_index],val_samples_gt[val_index],c,hi_margin)
            val_loss = hi_loss
    #        # print("{}\ttrain loss={:.4f}\t train OA={:.4f} val loss={:.4f}\t val OA={:.4f}".format(str(i + 1), trainloss, trainOA, valloss, valOA))
        loss =1* val_hi_loss
        if loss< best_loss :
            best_loss = loss
            torch.save(model.state_dict(), path_weight + r"model_metric.pt")
            print('save model...')
        # scheduler.step(valloss)
        torch.cuda.empty_cache()
        model.train()

    if i%10==0:
        print("{}\ttrain hi loss={:.4f}\t val hi loss={:.4f}\t".format(str(i + 1), hi_loss,val_hi_loss))
    torch.cuda.empty_cache()

save model...
1	train hi loss=4.8883	 val hi loss=4.9999	
save model...
11	train hi loss=3.8883	 val hi loss=4.9907	
save model...
21	train hi loss=2.8882	 val hi loss=4.8465	
save model...
31	train hi loss=2.3330	 val hi loss=4.4819	
save model...
41	train hi loss=1.9742	 val hi loss=3.7982	
save model...
51	train hi loss=1.7627	 val hi loss=2.7666	
save model...
61	train hi loss=1.5787	 val hi loss=2.3146	
save model...
71	train hi loss=1.4482	 val hi loss=2.1586	
save model...
81	train hi loss=1.3552	 val hi loss=2.1024	
save model...
91	train hi loss=1.2852	 val hi loss=2.0685	
save model...
101	train hi loss=1.2338	 val hi loss=2.0451	
save model...
111	train hi loss=1.1956	 val hi loss=2.0285	
save model...
121	train hi loss=1.1618	 val hi loss=2.0167	
save model...
131	train hi loss=1.1320	 val hi loss=2.0032	
save model...
141	train hi loss=1.1068	 val hi loss=1.9845	
save model...
151	train hi loss=1.0850	 val hi loss=1.9682	
save model...
161	train hi loss=1.0607	 val hi loss

In [17]:
with torch.no_grad():
    model.load_state_dict(torch.load(path_weight + r"model_metric.pt"))
    model.eval()
    #output_euc,output_hi,c= model(net_input)
    output_hi,c= model(net_input)
torch.cuda.empty_cache()
c =c.detach()

In [18]:
D_hi = utils.interclass_dis(output_hi[train_index],train_samples_gt[train_index],c,class_num)
D_hi = D_hi.to(torch.float64)
T = TreeRep.TreeRep(D_hi)
T.learn_tree()
Tree_stu = construt_tree.tree_structure(T,class_num)
# ===== 第一部分：训练标签合并 =====
a, b = Tree_stu.shape
# Tree_stu = np.concatenate(
#     [np.zeros((Tree_stu.shape[0], 1)), Tree_stu],
#     axis=1
# )
train_samples_gt = train_samples_gt.cpu().numpy()
merge_gt = train_samples_gt.copy()

for i in range(a - 1):
    child_nodes = np.where(Tree_stu[i + 1] == 1)[0] + 1

    if len(child_nodes) == 0:
        continue

    # 向量化替换（关键优化）
    mask = np.isin(train_samples_gt, child_nodes)
    merge_gt[mask] = i + 1

train_samples_gt = torch.from_numpy(train_samples_gt).to(device)


# ===== 第二部分：测试标签构建 =====
num, lab = Tree_stu.shape

samples_gt_merge = gt_reshape.copy()

createLabel = {}

for i in range(num - 1):
    row = Tree_stu[num - i - 1]

    child_nodes = np.where(row == 1)[0] + 1
    parent_node = np.where(row == 2)[0] + 1

    if len(child_nodes) == 0:
        continue

    # 向量化替换（关键优化）
    mask = np.isin(samples_gt_merge, child_nodes)
    samples_gt_merge[mask] = parent_node

    createLabel[f'testlabel_class_index{num - i - 1}'] = np.where(mask)
    createLabel[f'testlabel_class{num - i - 1}'] = samples_gt_merge.copy()

createLabel[f'testlabel_class{num}'] = gt_reshape.copy()

In [21]:
def three_branch_hicls(
        createLabel,
        device,
        max_epoch,
        learning_rate,
        path_weight,
        height,
        data,
        class_num,
        train_index,
        val_index,
        test_index,
        seed,
        load_model,
        c,
        layer,
        kernel_size
):

    # =========================================================
    # 初始化
    # =========================================================

    gt = torch.zeros(height * width, dtype=int).to(device)

    gt_fig = torch.zeros(height * width, dtype=int).to(device)

    root_children = np.where(Tree_stu[0] == 1)[0]

    # =========================================================
    # 新增：
    # 类别 -> 来源分类器映射
    # =========================================================

    parent_classifier_map = {}

    # =========================================================
    # 工具函数
    # =========================================================

    def get_node_info(node_idx):

        child_nodes = np.where(Tree_stu[node_idx] == 1)[0]

        parent_node = np.where(Tree_stu[node_idx] == 2)[0]

        coarse_nodes = child_nodes[child_nodes >= class_num]

        leaf_nodes = child_nodes[child_nodes < class_num]

        return child_nodes, parent_node, coarse_nodes, leaf_nodes

    def build_train_val_index(parent_id, label_key):

        train_sub_index = train_index[
            np.where(
                createLabel[label_key][train_index] == parent_id + 1
            )
        ]

        val_sub_index = val_index[
            np.where(
                createLabel[label_key][val_index] == parent_id + 1
            )
        ]

        return train_sub_index, val_sub_index

    def build_test_index(parent_model_idx, parent_id):

        parent_prediction = createLabel[
            'res_cla' + str(parent_model_idx)
        ]

        parent_test_index = createLabel[
            'test_index_cla' + str(parent_model_idx)
        ]

        sub_index = np.where(
            parent_prediction[parent_test_index].cpu() == parent_id
        )

        return parent_test_index[sub_index]

    # =========================================================
    # 主循环
    # =========================================================

    for node_idx in range(num):

        print(f'\n==============================')
        print(f'node = {node_idx}')

        child_nodes, parent_node, coarse_nodes, leaf_nodes = \
            get_node_info(node_idx)

        print('parent node:', parent_node)
        print('child node:', child_nodes)

        # =====================================================
        # Root Node
        # =====================================================

        if node_idx == 0:

            marix = utils.output_combine(
                Tree_stu,
                class_num,
                node_idx
            )

            (
                createLabel['OA_cla1'],
                createLabel['res_cla1']

            ) = base_class.train_node_classifier(

                device,
                max_epoch,
                learning_rate,
                path_weight,
                data,

                createLabel['testlabel_class1'],

                gt_reshape,

                train_index,
                val_index,
                test_index,

                train_index,
                val_index,
                test_index,

                seed,
                False,

                'model_metric',
                'model1',

                c,
                layer,
                hi_margin,
                euc_margin,
                0.5,

                True,

                kernel_size,
            )

            createLabel['test_index_cla1'] = test_index

            # =================================================
            # 更新最终结果
            # =================================================

            gt[test_index] = createLabel[
                'res_cla1'
            ][test_index]

            # =================================================
            # 建立 routing map
            # root 子类别由 model1 产生
            # =================================================

            for child in child_nodes:

                parent_classifier_map[child] = 1

            print('update parent_classifier_map:')
            print(parent_classifier_map)

            continue

        # =====================================================
        # 当前节点父类别
        # =====================================================

        parent_id = parent_node[0]

        # =====================================================
        # 通过 routing map 获取父分类器
        # =====================================================

        if parent_id not in parent_classifier_map:

            print(f'ERROR: parent_id {parent_id} '
                  f'not found in parent_classifier_map')

            continue

        parent_model_idx = parent_classifier_map[parent_id]

        print('parent_id:', parent_id)
        print('parent_model_idx:', parent_model_idx)

        # =====================================================
        # Debug:
        # 检查 parent_id 是否真的存在于父分类器输出
        # =====================================================

        parent_prediction = createLabel[
            'res_cla' + str(parent_model_idx)
        ]

        unique_parent_pred = torch.unique(
            parent_prediction
        ).cpu().numpy()

        print('unique parent prediction:',
              unique_parent_pred)

        if parent_id not in unique_parent_pred:

            print(f'WARNING: parent_id {parent_id} '
                  f'not in parent prediction')

        # =====================================================
        # 构建局部训练索引
        # =====================================================

        train_sub_index, val_sub_index = \
            build_train_val_index(
                parent_id,
                'testlabel_class' + str(node_idx)
            )

        # =====================================================
        # 构建测试索引
        # =====================================================

        test_sub_index = build_test_index(
            parent_model_idx,
            parent_id
        )

        print('train_sub_index num:',
              len(train_sub_index))

        print('val_sub_index num:',
              len(val_sub_index))

        print('test_sub_index num:',
              len(test_sub_index))

        # =====================================================
        # 防止空测试集
        # =====================================================

        if len(test_sub_index) == 0:

            print('WARNING: empty test_sub_index')
            continue

        # =====================================================
        # 保存索引
        # =====================================================

        createLabel[
            'train_index_cla' + str(node_idx + 1)
        ] = train_sub_index

        createLabel[
            'val_index_cla' + str(node_idx + 1)
        ] = val_sub_index

        createLabel[
            'test_index_cla' + str(node_idx + 1)
        ] = test_sub_index

        # =====================================================
        # 当前节点类别映射
        # =====================================================

        marix = child_nodes

        # =====================================================
        # 训练当前节点分类器
        # =====================================================

        (
            createLabel[
                'OA_cla' + str(node_idx + 1)
            ],

            createLabel[
                'res_cla' + str(node_idx + 1)
            ]

        ) = base_class.train_node_classifier(

            device,
            max_epoch,
            learning_rate,
            path_weight,
            data,

            createLabel[
                'testlabel_class' + str(node_idx + 1)
            ],

            gt_reshape,

            train_sub_index,
            val_sub_index,
            test_sub_index,

            train_index,
            val_index,
            test_index,

            seed,
            load_model,

            'model' + str(parent_model_idx),
            'model' + str(node_idx + 1),

            c,
            layer,

            hi_margin,
            euc_margin,
            0.5,

            True,

            kernel_size,
        )

        # =====================================================
        # 更新 routing map
        # 当前分类器负责产生 child_nodes
        # =====================================================

        for child in child_nodes:

            parent_classifier_map[child] = node_idx + 1

        print('update parent_classifier_map:')
        print(parent_classifier_map)

        # =====================================================
        # 更新最终结果
        # =====================================================

        gt[test_sub_index] = createLabel[
            'res_cla' + str(node_idx + 1)
        ][test_sub_index]

        # =====================================================
        # 可视化结果
        # =====================================================

        index_fig = torch.where(
            createLabel['res_cla1'] == parent_id
        )[0]

        gt_fig[index_fig] = createLabel[
            'res_cla' + str(node_idx + 1)
        ][index_fig]

    return gt

In [22]:
load_model =True
gt=three_branch_hicls(createLabel,device,500,0.0005,path_weight,height,data,class_num,train_index, val_index, test_index,seed,load_model,c,layer,kernel_size)
OA_hi = evaluate_performance(gt, test_samples_gt, test_gt_onehot, zeros)
OA_hi


node = 0
parent node: [9]
child node: [10 11 14]
Epoch: 0000 | Loss: 20.4093 | Val HI OA: 0.2000 | Val EUC OA: 0.5000
Epoch: 0010 | Loss: 18.6123 | Val HI OA: 0.5278 | Val EUC OA: 0.3667
Epoch: 0020 | Loss: 16.9726 | Val HI OA: 0.5556 | Val EUC OA: 0.3056
Epoch: 0030 | Loss: 15.4277 | Val HI OA: 0.5556 | Val EUC OA: 0.3111
Epoch: 0040 | Loss: 14.1894 | Val HI OA: 0.7833 | Val EUC OA: 0.3833
Epoch: 0050 | Loss: 13.2769 | Val HI OA: 0.7778 | Val EUC OA: 0.5611
Epoch: 0060 | Loss: 12.5124 | Val HI OA: 0.8444 | Val EUC OA: 0.6889
Epoch: 0070 | Loss: 11.6378 | Val HI OA: 0.9611 | Val EUC OA: 0.7611
Epoch: 0080 | Loss: 10.9386 | Val HI OA: 0.9556 | Val EUC OA: 0.7722
Epoch: 0090 | Loss: 10.2916 | Val HI OA: 0.9833 | Val EUC OA: 0.7778
Epoch: 0100 | Loss: 9.6959 | Val HI OA: 0.9944 | Val EUC OA: 0.7778
Epoch: 0110 | Loss: 9.1885 | Val HI OA: 0.9944 | Val EUC OA: 0.7778
Epoch: 0120 | Loss: 8.8141 | Val HI OA: 0.9944 | Val EUC OA: 0.7778
Epoch: 0130 | Loss: 8.5639 | Val HI OA: 0.9944 | Val EUC

tensor(0.1554, device='cuda:3')

In [27]:
OA_hi = evaluate_performance(gt, test_samples_gt, test_gt_onehot, zeros)
OA_hi

tensor(0.9586, device='cuda:3')